# json-extract-lora: the complete pipeline in ONE notebook (data -> train -> evaluate)
Builds the JSON-extraction adapter to the shared fragment standard and scores it against fair baselines.

**How to run**
1. Run the first four code cells (installs, environment check, standard, switches). Set `SMOKE_TEST = True` for the first pass: a tiny end-to-end dry run with `-smoke` file names. It exercises every stage, including the optional GCD step, and prints the time per training step.
2. Set `SMOKE_TEST = False` and run everything again (**Save Version -> Save & Run All** for the real run, so a disconnect cannot kill it).
3. After a disconnect, run all again: training resumes from its last checkpoint and evaluation resumes from `json_eval_results.jsonl`. Switch a stage off with `RUN_DATA / RUN_TRAIN / RUN_EVAL`; its files on disk are reused.

**Stages:** 1) clean ScrapeGraphAI-100k into train/val files and a data card. 2) train: all linear layers, r=16, alpha=32, 2 epochs, best checkpoint kept, registry entry + zip. 3) evaluate on a frozen topic-stratified paraloq set: `base`, `lora`, optional `lora_qkv` (old attention-only adapter), optional GCD versions; writes `json_reference.json`.

**Outputs (in /kaggle/working):** `json-extract-lora-adapter/` (+ `.zip`), `scrapegraph_clean_*.jsonl`, `data_card.json`, `json_eval_set_v2.json`, `json_eval_results.jsonl`, `json_reference.json`.


In [3]:
import sys, subprocess
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], capture_output=True, text=True)
    print("OK    " if r.returncode == 0 else "FAILED", pkgs)
    if r.returncode != 0: print(r.stderr[-1500:])

pip("bitsandbytes>=0.46.1", "accelerate", "peft", "datasets", "scikit-learn")
pip("transformers==5.0.0", "huggingface_hub>=1.3,<2")
pip("jsonschema", "safetensors")


OK     ('bitsandbytes>=0.46.1', 'accelerate', 'peft', 'datasets', 'scikit-learn')
OK     ('transformers==5.0.0', 'huggingface_hub>=1.3,<2')
OK     ('jsonschema', 'safetensors')


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must be set BEFORE torch is imported
import torch, transformers, huggingface_hub, peft, datasets, bitsandbytes, sklearn
print("transformers", transformers.__version__, "| hub", huggingface_hub.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__, "| bitsandbytes", bitsandbytes.__version__, "| sklearn", sklearn.__version__)
assert transformers.__version__.startswith("5."), f"transformers must be 5.x, got {transformers.__version__}"
from transformers.models.mistral.configuration_mistral import MistralConfig
assert torch.cuda.is_available(), "Enable a GPU: Kaggle > Settings > Accelerator"
assert torch.cuda.device_count() == 1, "CUDA_VISIBLE_DEVICES was set too late"
print("ALL GOOD")


transformers 5.0.0 | hub 1.29.0 | peft 0.20.0 | datasets 4.8.5 | bitsandbytes 0.50.2 | sklearn 1.6.1
ALL GOOD


In [5]:
import json, math, re, hashlib, os
import numpy as np

# ---------------------------------------------------------------------------------------------------
# THE FRAGMENT STANDARD. Edit it in ONE place, then paste this identical cell into EVERY notebook
# (tone, json, sql, legal, merge lab). HARD = must match for adapters to be mergeable.
# ---------------------------------------------------------------------------------------------------
FRAGMENT_STANDARD = {
    "schema": "fragment-standard/v1",
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    "base_revision": "63a8b081895390a26e140280378bc85ec8bce07a",
    "chat_template_sha256": "796853172235ab3aaa4810013ddfe2eab481bda3306dad4f700c00d436eed596",     # HARD (once pinned). Same: paste the value printed by the tone notebook
    "quantization": "bnb-nf4-double-quant, fp16-compute",
    "lora": {                         # HARD: r, alpha, target_modules, use_rslora, use_dora
        "r": 16, "alpha": 32, "dropout": 0.05, "bias": "none", "use_rslora": False, "use_dora": False,
        "target_modules": ["down_proj", "gate_proj", "k_proj", "o_proj", "q_proj", "up_proj", "v_proj"],
    },
    "training": {                     # SOFT: recorded, recommended identical, not merge-breaking
        "seed": 42, "lr": 2e-4, "scheduler": "linear", "warmup_frac": 0.03, "epochs": 2, "optimizer": "adamw_torch",
        "weight_decay": 0.01, "effective_batch": 16, "loss": "completion-only + EOS appended",
        "overlong": "drop, never truncate", "best_model": "load_best_model_at_end on eval_loss", "save_total_limit": 3,
    },
}
STD = FRAGMENT_STANDARD
SL = STD["lora"]
STD_SCALING = SL["alpha"] / SL["r"]
ATTN = {"q_proj", "k_proj", "v_proj", "o_proj"}
KEY_RE = re.compile(r"layers\.(\d+)\.(?:self_attn|mlp)\.(\w+_proj)\.lora_([AB])(?:\.\w+)?\.weight$")

def module_dims(hidden=4096, n_heads=32, n_kv=8, inter=14336):
    """module -> (in_features, out_features) for a Mistral/Llama-style block"""
    kv = hidden // n_heads * n_kv
    return {"q_proj": (hidden, hidden), "k_proj": (hidden, kv), "v_proj": (hidden, kv), "o_proj": (hidden, hidden),
            "gate_proj": (hidden, inter), "up_proj": (hidden, inter), "down_proj": (inter, hidden)}

def peft_key(layer, module, ab):
    blk = "self_attn" if module in ATTN else "mlp"
    return f"base_model.model.model.layers.{layer}.{blk}.{module}.lora_{ab}.weight"

def compat_key(base, base_rev, r, alpha, targets, rslora, dora, tmpl_sha, quant=STD["quantization"]):
    s = json.dumps([base, base_rev, r, alpha, sorted(targets), bool(rslora), bool(dora), tmpl_sha, quant])
    return hashlib.sha256(s.encode()).hexdigest()[:16]

def check_core(cfg, shapes, reg=None, dims=None, n_layers=32, std=STD):
    """Pure function (no GPU / file IO). Returns (rows, verdict). status: OK | ALIGNABLE | FAIL | UNKNOWN"""
    L = std["lora"]; dims = dims or module_dims(); reg = reg or {}; rows = []
    add = lambda name, status, detail="": rows.append(("HARD", name, status, str(detail)))
    ok = lambda c: "OK" if c else "FAIL"
    add("peft_type is LORA", ok(cfg.get("peft_type") == "LORA"), cfg.get("peft_type"))
    add("base model", ok(cfg.get("base_model_name_or_path") == std["base_model"]), cfg.get("base_model_name_or_path"))
    r, alpha, rs = cfg.get("r"), cfg.get("lora_alpha"), bool(cfg.get("use_rslora", False))
    scaling = alpha / (math.sqrt(r) if rs else r)
    add("rank r", "OK" if r == L["r"] else "ALIGNABLE", f"{r} (standard {L['r']})" + (" - larger: lossy SVD truncation" if r > L["r"] else ""))
    add("lora_alpha / scaling", "OK" if (alpha == L["alpha"] and r == L["r"] and not rs) else "ALIGNABLE", f"alpha {alpha}, scaling {scaling:g} (standard {STD_SCALING:g})")
    tm = cfg.get("target_modules")
    if isinstance(tm, (list, set, tuple)):
        tms, std_t = set(tm), set(L["target_modules"])
        add("target_modules", "OK" if tms == std_t else ("FAIL" if tms - std_t else "ALIGNABLE"), sorted(tms))
    else:
        add("target_modules", "FAIL", f"regex/str target_modules not supported: {tm!r}")
    add("use_dora", ok(not cfg.get("use_dora", False)))
    add("use_rslora", "OK" if rs == L["use_rslora"] else "ALIGNABLE", rs)
    add("no rank/alpha patterns", ok(not cfg.get("rank_pattern") and not cfg.get("alpha_pattern")))
    add("no modules_to_save (extra full weights)", ok(not cfg.get("modules_to_save")), cfg.get("modules_to_save"))
    bad = [k for k in shapes if not KEY_RE.search(k)]
    add("only LoRA A/B tensors", ok(not bad), bad[:3])
    shape_bad, present = [], set()
    for k, shp in shapes.items():
        m = KEY_RE.search(k)
        if not m: continue
        layer, mod, ab = int(m.group(1)), m.group(2), m.group(3)
        if mod not in dims: shape_bad.append((k, "unknown module")); continue
        din, dout = dims[mod]
        want = (r, din) if ab == "A" else (dout, r)
        if tuple(shp) != want: shape_bad.append((k, tuple(shp), want))
        present.add((layer, mod, ab))
    add("tensor shapes", ok(not shape_bad), shape_bad[:2])
    full = {(l, m, ab) for l in range(n_layers) for m in L["target_modules"] for ab in "AB"}
    add("coverage (all layers x modules)", "OK" if full <= present else "ALIGNABLE", f"{len(present & full)}/{len(full)} tensors")
    for fld, label in (("base_revision", "base revision"), ("chat_template_sha256", "chat template hash")):
        want, got = std.get(fld), reg.get(fld)
        if want is None or got is None: add(label, "UNKNOWN", f"standard={want} adapter={got}")
        else: add(label, ok(want == got), f"{got}")
    st = [x[2] for x in rows]
    verdict = ("INCOMPATIBLE" if "FAIL" in st else "NEEDS ALIGNMENT" if "ALIGNABLE" in st
               else "COMPATIBLE (some checks unverified)" if "UNKNOWN" in st else "MERGE-READY")
    return rows, verdict

def print_report(name, rows, verdict):
    print(f"\n=== {name} ===")
    for lvl, n, s, d in rows: print(f"  [{s:<9}] {n:<42} {d[:90]}")
    print(f"  -> VERDICT: {verdict}")

def check_adapter_dir(path, dims=None, n_layers=32, std=STD, verbose=True, framework="pt"):
    from safetensors import safe_open
    cfg = json.load(open(f"{path}/adapter_config.json"))
    rp = f"{path}/registry_entry.json"
    reg = json.load(open(rp)) if os.path.exists(rp) else {}
    with safe_open(f"{path}/adapter_model.safetensors", framework=framework) as sf:
        shapes = {k: tuple(sf.get_slice(k).get_shape()) for k in sf.keys()}
    rows, verdict = check_core(cfg, shapes, reg, dims, n_layers, std)
    if verbose: print_report(os.path.basename(path.rstrip("/")), rows, verdict)
    return rows, verdict

def check_set(paths, dims=None, n_layers=32):
    """Every adapter that will be merged together must pass check_adapter_dir AND share one compatibility_key."""
    keys, verdicts = {}, {}
    for p in paths:
        rows, verdicts[p] = check_adapter_dir(p, dims, n_layers)
        rp = f"{p}/registry_entry.json"
        keys[p] = json.load(open(rp)).get("compatibility_key") if os.path.exists(rp) else None
    print("\ncompatibility keys:", keys)
    same = len({k for k in keys.values() if k}) <= 1 and all(keys.values())
    print("ALL SHARE ONE compatibility_key:", "YES" if same else "NO / missing  <- fix before merging")
    return verdicts, same

def lora_delta_norms(path, scaling=STD_SCALING):
    """Frobenius norm of each module's delta  s*B@A  (computed through the r x r Gram matrices; never builds the dense delta)."""
    import torch
    from safetensors.torch import load_file
    t = load_file(f"{path}/adapter_model.safetensors"); pairs = {}
    for k, v in t.items():
        m = KEY_RE.search(k)
        if m: pairs.setdefault((int(m.group(1)), m.group(2)), {})[m.group(3)] = v.float()
    norms = {}
    for (l, mod), ab in pairs.items():
        A, B = ab["A"], ab["B"]
        norms[f"{l}.{mod}"] = float(scaling * torch.sqrt(torch.clamp(torch.trace((B.T @ B) @ (A @ A.T)), min=0)))
    v = np.array(list(norms.values()))
    return {"mean": float(v.mean()), "max": float(v.max()), "l2_total": float(np.sqrt((v ** 2).sum())), "n_modules": len(v)}

# ------------------------------- aligning an EXTERNAL adapter to the standard -------------------------------
def align_module(A, B, r_ext, alpha_ext, rslora, r_std=SL["r"], alpha_std=SL["alpha"]):
    """Re-express  delta = s_ext * B@A  as  s_std * B2@A2  with rank r_std. Exact if r_ext <= r_std, best rank-r_std approx otherwise."""
    s_ext = alpha_ext / (math.sqrt(r_ext) if rslora else r_ext); s_std = alpha_std / r_std
    A = A.astype(np.float32); B = B.astype(np.float32)
    if r_ext <= r_std:
        pad = r_std - r_ext
        A2 = np.vstack([A, np.zeros((pad, A.shape[1]), np.float32)])
        B2 = np.hstack([B * (s_ext / s_std), np.zeros((B.shape[0], pad), np.float32)])
        return A2, B2, 0.0
    Qb, Rb = np.linalg.qr(B)                 # B: out x r_ext
    Qa, Ra = np.linalg.qr(A.T)               # A.T: in x r_ext
    U, S, Vt = np.linalg.svd(s_ext * (Rb @ Ra.T))
    err = float(np.sqrt((S[r_std:] ** 2).sum() / max((S ** 2).sum(), 1e-30)))
    Ur, Sr, Vtr = U[:, :r_std], S[:r_std], Vt[:r_std]
    f = np.sqrt(Sr / s_std)
    return (f[:, None] * (Vtr @ Qa.T)).astype(np.float32), ((Qb @ Ur) * f).astype(np.float32), err

def align_tensors(tensors, cfg, dims=None, n_layers=32, std=STD):
    dims = dims or module_dims(); L = std["lora"]
    r_ext, a_ext, rs = cfg["r"], cfg["lora_alpha"], bool(cfg.get("use_rslora", False))
    pairs = {}
    for k, v in tensors.items():
        m = KEY_RE.search(k)
        if m: pairs.setdefault((int(m.group(1)), m.group(2)), {})[m.group(3)] = np.asarray(v, dtype=np.float32)
    out, errs = {}, []
    for layer in range(n_layers):
        for mod in L["target_modules"]:
            din, dout = dims[mod]; ab = pairs.get((layer, mod))
            if ab and "A" in ab and "B" in ab:
                A2, B2, e = align_module(ab["A"], ab["B"], r_ext, a_ext, rs, L["r"], L["alpha"]); errs.append(e)
            else:                                           # module the external adapter did not train -> explicit zero delta
                A2, B2 = np.zeros((L["r"], din), np.float32), np.zeros((dout, L["r"]), np.float32)
            out[peft_key(layer, mod, "A")], out[peft_key(layer, mod, "B")] = A2, B2
    n_all = n_layers * len(L["target_modules"])
    return out, {"max_rel_error": max(errs) if errs else 0.0, "modules_converted": len(errs), "modules_zero_filled": n_all - len(errs)}

def align_adapter(src_dir, dst_dir, dims=None, n_layers=32):
    """Convert a compatible-base external adapter into the standard format. Refuses INCOMPATIBLE adapters."""
    import torch
    from safetensors.torch import load_file, save_file
    from peft import LoraConfig
    rows, verdict = check_adapter_dir(src_dir, dims, n_layers)
    if verdict == "INCOMPATIBLE":
        raise ValueError("Cannot align: " + "; ".join(f"{n}: {d}" for _, n, s, d in rows if s == "FAIL"))
    cfg = json.load(open(f"{src_dir}/adapter_config.json"))
    t = {k: v.float().numpy() for k, v in load_file(f"{src_dir}/adapter_model.safetensors").items()}
    out, rep = align_tensors(t, cfg, dims, n_layers)
    os.makedirs(dst_dir, exist_ok=True)
    save_file({k: torch.from_numpy(v).contiguous() for k, v in out.items()}, f"{dst_dir}/adapter_model.safetensors")
    LoraConfig(r=SL["r"], lora_alpha=SL["alpha"], lora_dropout=SL["dropout"], bias=SL["bias"], target_modules=SL["target_modules"],
               use_rslora=False, use_dora=False, task_type="CAUSAL_LM", base_model_name_or_path=STD["base_model"]).save_pretrained(dst_dir)
    h = hashlib.sha256(open(f"{src_dir}/adapter_model.safetensors", "rb").read()).hexdigest()
    json.dump({"registry_schema": "fragment-registry/v1", "fragment_id": os.path.basename(src_dir.rstrip("/")) + "-aligned",
               "aligned_from": {"path": src_dir, "r": cfg["r"], "lora_alpha": cfg["lora_alpha"], "target_modules": cfg.get("target_modules"),
                                "use_rslora": cfg.get("use_rslora", False), "source_sha256": h},
               "alignment": rep, "lossless": rep["max_rel_error"] == 0.0,
               "base_model": STD["base_model"], "lora_scaling": STD_SCALING,
               "base_revision_verified": False,   # we cannot verify which exact base commit an external adapter was trained on
               "compatibility_key": (compat_key(STD["base_model"], STD["base_revision"], SL["r"], SL["alpha"], SL["target_modules"], False, False,
                                                STD["chat_template_sha256"]) if STD["base_revision"] and STD["chat_template_sha256"] else None)},
              open(f"{dst_dir}/registry_entry.json", "w"), indent=2)
    print("aligned ->", dst_dir, rep)
    return rep

def stamp_registry(path, base_revision, chat_template_sha256, fragment_id=None):
    """For an adapter that already exists (e.g. the json adapter): add the fields the merge system needs, computed from its OWN adapter_config."""
    cfg = json.load(open(f"{path}/adapter_config.json"))
    rp = f"{path}/registry_entry.json"
    reg = json.load(open(rp)) if os.path.exists(rp) else {}
    tm = cfg.get("target_modules")
    reg.update({"registry_schema": "fragment-registry/v1", "base_model": cfg.get("base_model_name_or_path"),
                "base_revision": base_revision, "chat_template_sha256": chat_template_sha256,
                "lora": {"r": cfg.get("r"), "alpha": cfg.get("lora_alpha"), "target_modules": sorted(tm) if isinstance(tm, (list, set)) else tm,
                         "use_rslora": bool(cfg.get("use_rslora", False)), "use_dora": bool(cfg.get("use_dora", False))},
                "compatibility_key": compat_key(cfg.get("base_model_name_or_path"), base_revision, cfg.get("r"), cfg.get("lora_alpha"),
                                                tm if isinstance(tm, (list, set)) else [], cfg.get("use_rslora", False), cfg.get("use_dora", False), chat_template_sha256)})
    if fragment_id: reg["fragment_id"] = fragment_id
    if os.path.exists(f"{path}/adapter_model.safetensors"):
        reg["adapter_size_bytes"] = os.path.getsize(f"{path}/adapter_model.safetensors")
        reg["sha256"] = hashlib.sha256(open(f"{path}/adapter_model.safetensors", "rb").read()).hexdigest()
    json.dump(reg, open(rp, "w"), indent=2)
    return reg["compatibility_key"]

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SMOKE_TEST = False                   # True = tiny dry run of ALL stages (files get a "-smoke" suffix). False = the real run.
RUN_DATA, RUN_TRAIN, RUN_EVAL = True, True, True      # switch a stage off to skip it; its files on disk are reused
SUFFIX = "-smoke" if SMOKE_TEST else ""
print("SMOKE_TEST:", SMOKE_TEST, "| stages:", dict(data=RUN_DATA, train=RUN_TRAIN, eval=RUN_EVAL))

# ---- the prompt contract: ONE definition used by every stage ----
DEFAULT_INSTRUCTION = "Extract all the information that the schema asks for."
TEMPLATE = ("Extract the information from the text as JSON that follows the schema.\n"
            "Output JSON only, no explanation.\n\n"
            "Instruction: {instruction}\n\n"
            "Schema:\n{schema}\n\nText:\n{content}")
# one consistent way to say "this field is absent": teacher models write many variants of "N/A"; we normalise them to ""
PLACEHOLDERS = {"n/a", "na", "none", "null", "unknown", "not specified", "not available", "not provided", "not mentioned",
                "unspecified", "not applicable", "-", "--", "nil"}


# ---- prompt contract (recorded in registry for the router) ----
PROMPT_CONTRACT = {
    "template": TEMPLATE,
    "fields": ["instruction", "schema", "content"],
    "default_instruction": DEFAULT_INSTRUCTION,
    "output_format": "raw JSON object, no wrapper text",
}


SMOKE_TEST: False | stages: {'data': True, 'train': True, 'eval': True}


## Stage 1: data (cleaning, token-aware, with a data card)

In [18]:
if RUN_DATA:
    import os, json, re, hashlib, random, math, datetime
    from collections import Counter, defaultdict
    import numpy as np
    from jsonschema import validate, ValidationError
    from transformers import AutoTokenizer

    SEED = STD["training"]["seed"]
    MAX_LEN = 1536                       # training cap in TOKENS (prompt + answer). Longer rows are dropped, never truncated.
    OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    DATASET = "scrapegraphai/scrapegraphai-100k"

    CFG = dict(
        target_rows=6000, scan_limit=200_000,
        min_content=200, max_content=6000,     # characters, a cheap pre-filter. The real limit is MAX_LEN tokens (checked below).
        max_schema_chars=2000, max_resp_depth=4, max_instruction_chars=300,
        min_leaves=1, min_grounded=0.7, min_ascii=0.9,
        max_per_schema=20, max_per_family=25, val_schema_frac=0.06,
    )
    if SMOKE_TEST: CFG.update(target_rows=500, scan_limit=60_000)
    # blocked / captcha pages announce themselves at the TOP of the page. Weak phrases such as "enable javascript"
    # are normal footer boilerplate on real pages, so they are NOT used.
    JUNK_STRONG = ["not a robot", "captcha", "verify you are human", "verifying you are human", "are you a human", "access denied",
                   "unusual traffic", "you have been blocked", "please enable cookies", "enable javascript and cookies",
                   "security of your connection", "ray id", "just a moment", "attention required"]
    JUNK_WINDOW = 1500
    MODEL_ALLOW = None        # e.g. ["gpt-4", "claude", "gemini"] AFTER you have read the model counts printed below


    tokenizer = AutoTokenizer.from_pretrained(STD["base_model"], revision=STD["base_revision"])
    tmpl_sha = hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest()
    assert tmpl_sha == STD["chat_template_sha256"], "chat template differs from the fragment standard"

    def render_prompt(text):
        return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)

    def n_tokens(prompt_text, completion):
        p = len(tokenizer(render_prompt(prompt_text), add_special_tokens=False)["input_ids"])
        c = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
        return p + c + 1                  # +1 for EOS
    print("config ok | template hash matches the standard")


config ok | template hash matches the standard


In [19]:
if RUN_DATA:
    def norm(s): return re.sub(r"\s+", " ", str(s)).strip().lower()
    def ascii_ratio(s): return sum(c.isascii() for c in s) / max(1, len(s))
    def is_placeholder(v): return isinstance(v, str) and norm(v) in PLACEHOLDERS
    def leaves(o):
        if isinstance(o, dict):
            for v in o.values(): yield from leaves(v)
        elif isinstance(o, list):
            for v in o: yield from leaves(v)
        elif o not in (None, "", 0): yield o        # 0 / False / "" / None are not content
    def depth(o):
        if isinstance(o, dict): return 1 + max((depth(v) for v in o.values()), default=0)
        if isinstance(o, list): return 1 + max((depth(v) for v in o), default=0)
        return 0

    def blank_placeholders(o):
        if isinstance(o, dict): return {k: blank_placeholders(v) for k, v in o.items()}
        if isinstance(o, list): return [blank_placeholders(v) for v in o]
        return "" if is_placeholder(o) else o

    # ---- write the answer in the SCHEMA'S property order (not alphabetical) ----
    # Schema-constrained decoding builds its grammar in schema order. If the adapter learned a different order, the grammar forces it
    # off the path it learned. Training on schema order removes that conflict.
    def _deref(s, defs):
        for _ in range(10):
            if isinstance(s, dict) and "$ref" in s: s = defs.get(s["$ref"].split("/")[-1], {})
            else: break
        return s if isinstance(s, dict) else {}
    def _branch(s, o, defs):
        s = _deref(s, defs)
        for key in ("anyOf", "oneOf", "allOf"):
            for b in s.get(key, []):
                b = _deref(b, defs)
                if (isinstance(o, dict) and "properties" in b) or (isinstance(o, list) and (b.get("type") == "array" or "items" in b)): return b
        return s
    def reorder(o, s, defs):
        s = _branch(s, o, defs)
        if isinstance(o, dict):
            props = s.get("properties", {})
            out = {k: reorder(o[k], props.get(k, {}), defs) for k in props if k in o}
            out.update({k: v for k, v in o.items() if k not in out})        # keys outside the schema, if any, go last
            return out
        if isinstance(o, list): return [reorder(v, s.get("items", {}), defs) for v in o]
        return o

    def check(row):
        # returns (clean_row, None) or (None, reason). Cheap checks first, token counting last.
        if not row.get("response_is_valid"): return None, "flag invalid"
        c = row.get("content") or ""
        if not (CFG["min_content"] <= len(c) <= CFG["max_content"]): return None, "content length"
        if len(row["schema"]) > CFG["max_schema_chars"]: return None, "schema too big"
        if MODEL_ALLOW and not any(m in (row.get("llm_model") or "") for m in MODEL_ALLOW): return None, "model not allowed"
        if any(k in c[:JUNK_WINDOW].lower() for k in JUNK_STRONG): return None, "junk page"
        if ascii_ratio(c) < CFG["min_ascii"]: return None, "not english"
        ins = re.sub(r"\s+", " ", row.get("prompt") or "").strip()
        if ins and ascii_ratio(ins) < CFG["min_ascii"]: return None, "instruction not english"      # an empty instruction falls back to the default
        if len(ins) > CFG["max_instruction_chars"]: return None, "instruction too long"
        ins = ins or DEFAULT_INSTRUCTION
        try: schema, resp = json.loads(row["schema"]), json.loads(row["response"])
        except Exception: return None, "unparseable json"
        if not isinstance(resp, dict): return None, "response not object"
        # Catch blocked / canned scraper error responses
        if resp.get("certainty") == 0 or resp.get("certainty_percentage") == 0: return None, "blocked page error"
        if "chapters" in resp and len(resp.get("chapters", [])) == 0 and len(resp) <= 2: return None, "empty chapter error"
        # Reject extra keys not in schema properties
        props = schema.get("properties")
        if isinstance(props, dict) and props and any(k not in props for k in resp.keys()):
            return None, "keys not in schema"
        try: validate(resp, schema)
        except ValidationError: return None, "fails schema"
        except Exception: return None, "bad schema"
        blanked = blank_placeholders(resp)
        try: validate(blanked, schema)
        except Exception: blanked = resp                                    # blanking would break the schema (e.g. an enum): keep the original
        did_blank = blanked != resp
        resp = blanked
        if depth(resp) > CFG["max_resp_depth"]: return None, "too deep"
        ls = list(leaves(resp))
        if len(ls) < CFG["min_leaves"]: return None, "too empty"
        strs = [norm(v) for v in ls if isinstance(v, str) and len(v) >= 4]
        cn = norm(c)
        if strs and sum(s in cn for s in strs) / len(strs) < CFG["min_grounded"]: return None, "not grounded in text"
        defs = schema.get("$defs") or schema.get("definitions") or {}
        schema_c = json.dumps(schema, separators=(",", ":"))
        completion = json.dumps(reorder(resp, schema, defs), ensure_ascii=False)
        prompt = TEMPLATE.format(instruction=ins, schema=schema_c, content=c)
        nt = n_tokens(prompt, completion)
        if nt > MAX_LEN: return None, f"over {MAX_LEN} tokens"
        props_keys = sorted(props.keys()) if isinstance(props, dict) else []
        schema_family = hashlib.md5(",".join(props_keys).encode()).hexdigest()[:12]
        return {"prompt": prompt, "completion": completion, "schema_hash": row["schema_hash"],
                "schema_family": schema_family, "model": row.get("llm_model"),
                "source": row.get("source"), "n_tokens": nt, "blanked": did_blank}, None
    print("filters ready")


filters ready


In [20]:
if RUN_DATA:
    from datasets import load_dataset
    stream = (load_dataset(DATASET, split="train", streaming=True)
              .filter(lambda r: r["response_is_valid"] and r["content"]
                      and CFG["min_content"] <= len(r["content"]) <= CFG["max_content"])
              .shuffle(seed=SEED, buffer_size=500 if SMOKE_TEST else 5_000))

    kept, why, per_schema, per_family, seen = [], Counter(), Counter(), Counter(), set()
    n_scanned = 0
    for row in stream:
        n_scanned += 1
        if n_scanned > CFG["scan_limit"] or len(kept) >= CFG["target_rows"]: break
        out, reason = check(row)
        if reason: why[reason] += 1; continue
        if per_schema[out["schema_hash"]] >= CFG["max_per_schema"]: why["schema cap"] += 1; continue
        if per_family[out["schema_family"]] >= CFG.get("max_per_family", 10): why["schema family cap"] += 1; continue
        key = hashlib.md5(norm(out["prompt"].split("Text:\n", 1)[1])[:1000].encode()).hexdigest()
        if key in seen: why["duplicate content"] += 1; continue
        seen.add(key); per_schema[out["schema_hash"]] += 1; per_family[out["schema_family"]] += 1; kept.append(out)
        if len(kept) % 500 == 0: print(f"  kept {len(kept)} after scanning {n_scanned}", flush=True)
    model_counts = Counter(k["model"] for k in kept)
    print("rows with placeholders normalised:", sum(k["blanked"] for k in kept))
    print(f"scanned {n_scanned} (after the cheap pre-filter) | kept {len(kept)} | schemas {len(per_schema)}")
    print("rejected:", dict(why.most_common()))
    print("top models:", model_counts.most_common(10))


rows with placeholders normalised: 56
scanned 25865 (after the cheap pre-filter) | kept 193 | schemas 158
rejected: {'junk page': 11676, 'blocked page error': 8453, 'over 1536 tokens': 1818, 'not grounded in text': 799, 'instruction too long': 773, 'empty chapter error': 552, 'schema too big': 471, 'too empty': 446, 'schema family cap': 397, 'not english': 185, 'duplicate content': 62, 'instruction not english': 22, 'keys not in schema': 12, 'too deep': 5, 'response not object': 1}
top models: [('gpt-4o-mini', 116), ('gpt-4o', 24), ('llama3.1', 6), ('llama3.2', 5), (None, 4), ('gemma3:27b', 4), ('mistral-large-latest', 3), ('llama3', 3), ('llama3-8b-8192', 2), ('qwen2.5:14b-instruct-q5_K_M', 2)]


In [21]:
if RUN_DATA:
    families = sorted({k["schema_family"] for k in kept}); random.Random(SEED).shuffle(families)
    # THREE-WAY SPLIT BY SCHEMA FAMILY: val and test evaluate generalization on UNSEEN schema families
    n_val  = max(1, int(len(families) * CFG["val_schema_frac"]))
    n_test = max(1, int(len(families) * CFG.get("test_schema_frac", 0.06)))
    val_f  = set(families[:n_val])
    test_f = set(families[n_val:n_val + n_test])
    strip = lambda k: {"prompt": k["prompt"], "completion": k["completion"], "schema_hash": k["schema_hash"], "schema_family": k["schema_family"]}
    train = [strip(k) for k in kept if k["schema_family"] not in val_f and k["schema_family"] not in test_f]
    val   = [strip(k) for k in kept if k["schema_family"] in val_f]
    test  = [strip(k) for k in kept if k["schema_family"] in test_f]
    assert not ({e["schema_family"] for e in train} & {e["schema_family"] for e in val}), "schema family leak train-val"
    assert not ({e["schema_family"] for e in train} & {e["schema_family"] for e in test}), "schema family leak train-test"
    assert not ({e["schema_family"] for e in val} & {e["schema_family"] for e in test}), "schema family leak val-test"
    print("train:", len(train), "| val:", len(val), "| test:", len(test), "| val families:", len(val_f), "| test families:", len(test_f))

    for name, data in [("train", train), ("val", val), ("test", test)]:
        with open(f"{OUT}/scrapegraph_clean_{name}{SUFFIX}.jsonl", "w", encoding="utf-8") as f:
            for e in data: f.write(json.dumps(e, ensure_ascii=False) + "\n")

    try:
        from huggingface_hub import dataset_info
        ds_rev = dataset_info(DATASET).sha
    except Exception as e:
        ds_rev = None; print("could not read dataset revision:", repr(e))
    card = {"dataset": DATASET, "dataset_revision": ds_rev, "base_model": STD["base_model"], "base_revision": STD["base_revision"],
            "chat_template_sha256": tmpl_sha, "template": TEMPLATE, "default_instruction": DEFAULT_INSTRUCTION, "rows_placeholders_blanked": int(sum(k["blanked"] for k in kept)), "max_len": MAX_LEN, "seed": SEED, "cfg": CFG,
            "junk_strong": JUNK_STRONG, "junk_window": JUNK_WINDOW, "model_allow": MODEL_ALLOW,
            "n_scanned": n_scanned, "n_kept": len(kept), "n_train": len(train), "n_val": len(val), "n_test": len(test),
            "val_schema_frac": CFG["val_schema_frac"], "test_schema_frac": CFG.get("test_schema_frac", 0.06),
            "rejected": dict(why), "top_models": model_counts.most_common(15),
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
    json.dump(card, open(f"{OUT}/data_card{SUFFIX}.json", "w"), indent=2)
    print("saved: scrapegraph_clean_train.jsonl, scrapegraph_clean_val.jsonl, data_card.json")


train: 181 | val: 7 | test: 5 | val families: 5 | test families: 5
saved: scrapegraph_clean_train.jsonl, scrapegraph_clean_val.jsonl, data_card.json


In [22]:
if RUN_DATA:
    T_ = np.array([k["n_tokens"] for k in kept])
    print(f"tokens/example: mean {T_.mean():.0f} | median {np.median(T_):.0f} | p95 {np.percentile(T_, 95):.0f} | max {T_.max()}")
    if len(kept) < 2000 and not SMOKE_TEST:
        print(f"\nWARNING: only {len(kept)} rows kept (target >= 2000). Read the 'rejected' counts above and loosen the biggest one:")
        print("  'too empty' -> min_leaves 1 | 'not grounded' -> min_grounded 0.6 | 'over N tokens' -> raise MAX_LEN (also in notebook 2)")
    for e in random.Random(0).sample(train, 5):
        print("\nSCHEMA :", e["prompt"].split("Schema:\n")[1].split("\n\nText:")[0][:200])
        print("TEXT   :", e["prompt"].split("Text:\n")[1][:300].replace("\n", " "))
        print("ANSWER :", e["completion"][:300]); print("-" * 60)
    print("\nREAD THE 5 EXAMPLES ABOVE: does each answer follow from its text?")


    # ---- label audit: what does this data teach about ABSENT fields? ----
    def top_cov(e):
        schema = json.loads(e["prompt"].split("\n\nSchema:\n", 1)[1].split("\n\nText:\n", 1)[0])
        props = list((schema.get("properties") or {}).keys()); resp = json.loads(e["completion"])
        return sum(1 for p in props if resp.get(p) not in (None, "", [], {})) / max(1, len(props))
    cov = np.array([top_cov(e) for e in train])
    print(f"\nfield coverage (filled top-level fields / schema fields): mean {cov.mean():.2f} | rows with an empty field: {(cov < 1).mean():.0%}")
    print(f"rows where N/A-style placeholders were normalised to empty: {np.mean([k['blanked'] for k in kept]):.0%}")
    print("distinct sources among kept rows:", len({k.get('source') for k in kept}), "| most common:", Counter(k.get('source') for k in kept).most_common(3))
    print("sample instructions:", [e["prompt"].split("Instruction: ", 1)[1].split("\n\nSchema:")[0][:90] for e in random.Random(1).sample(train, 4)])
    if cov.mean() > 0.97 and not SMOKE_TEST:
        print("NOTE: almost every kept row has every field filled. The adapter will rarely see how to leave a field empty; expect invented values for absent fields.")

    # ---- degenerate & blocked-signature safety audit ----
    def is_blocked_signature(e):
        try:
            resp = json.loads(e['completion'])
            # Detect blocked page artifacts (e.g. certainty_percentage: 0, empty chapters)
            if resp.get("certainty_percentage") == 0: return True
            if "chapters" in resp and len(resp["chapters"]) == 0 and len(resp) <= 2: return True
            # Detect borderline answers with fewer than 2 meaningful string/numeric values
            meaningful = [v for v in leaves(resp) if isinstance(v, (int, float)) or (isinstance(v, str) and len(v.strip()) >= 2)]
            return len(meaningful) < 1
        except Exception: return True

    blocked_share = np.mean([is_blocked_signature(e) for e in train])
    print(f"degenerate / blocked-signature share in train: {blocked_share:.1%}")
    assert blocked_share < 0.05, f"Degenerate / blocked answers {blocked_share:.1%} >= 5% threshold! Check junk filtering."


tokens/example: mean 1049 | median 1154 | p95 1504 | max 1535

  'too empty' -> min_leaves 1 | 'not grounded' -> min_grounded 0.6 | 'over N tokens' -> raise MAX_LEN (also in notebook 2)

SCHEMA : {"$defs":{"ItemModel":{"properties":{"startup_name":{"title":"Startup Name","type":"string"},"founder_names":{"title":"Founder Names","type":"string"},"year_founded":{"title":"Year Founded","type":"st
TEXT   : [ 0 ](/cart) Skip to Content [ ![Exceptional Startups](//images.squarespace- cdn.com/content/v1/659495743dc28573a0e396fc/f5d31cea-f092-4c08-80a2-26e82cddf835/startup.png?format=1500w) ](/) [ ![Exceptional Startups](//images.squarespace- cdn.com/content/v1/659495743dc28573a0e396fc/4535db8e-d724-4ba8-
ANSWER : {"items": [], "urls": ["/list-2025"]}
------------------------------------------------------------

SCHEMA : {"$defs":{"Project":{"properties":{"title":{"description":"The title of the product in English","title":"Title","type":"string"},"description":{"description":"Description and 

## Stage 2: training (fragment standard: all linear layers, r=16, alpha=32)

In [24]:
if RUN_TRAIN:
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    import json, re, math, time, hashlib, random, datetime, shutil, inspect, glob
    from collections import Counter
    import numpy as np
    import torch
    assert torch.cuda.is_available(), "Enable a GPU: Kaggle > Settings > Accelerator"
    assert torch.cuda.device_count() == 1, "restart the kernel and run the first two cells again"

    FRAGMENT_ID  = "json-extract-lora"
    BASE_MODEL   = STD["base_model"]
    BASE_REVISION = STD["base_revision"]
    SEED         = STD["training"]["seed"]
    EPOCHS       = 1 if SMOKE_TEST else STD["training"]["epochs"]
    BATCH, ACCUM = 1, 16                                   # effective batch 16 = standard; batch 1 avoids padding and fits a T4
    assert BATCH * ACCUM == STD["training"]["effective_batch"]
    MAX_LEN      = 1536
    NUM_TRAIN_EXAMPLES = 96 if SMOKE_TEST else None        # None = all rows from notebook 1
    VAL_N        = 16 if SMOKE_TEST else 100
    MAX_STEPS    = 6  if SMOKE_TEST else -1
    EVAL_EVERY   = 2  if SMOKE_TEST else 25                # eval_steps must equal save_steps for load_best_model_at_end

    OUT_ROOT    = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    SUFFIX      = "-smoke" if SMOKE_TEST else ""
    ADAPTER_DIR = f"{OUT_ROOT}/json-extract-lora-adapter{SUFFIX}"
    CKPT_DIR    = f"{OUT_ROOT}/json-extract-lora-ckpt{SUFFIX}"
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    def find_file(name):
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"{OUT_ROOT}/{name}")
        assert hits, f"{name} not found. Attach notebook 1's output via Add Input, or run notebook 1 first."
        return hits[0]
    print("config ok | smoke:", SMOKE_TEST)


config ok | smoke: False


In [25]:
if RUN_TRAIN:
    def read_jsonl(p):
        with open(p, encoding="utf-8") as f: return [json.loads(l) for l in f]
    card = json.load(open(find_file(f"data_card{SUFFIX}.json")))
    assert card["template"] == TEMPLATE, "prompt template differs from the one the data was built with"
    assert card["base_revision"] == STD["base_revision"] and card["chat_template_sha256"] == STD["chat_template_sha256"], "data card was built against a different base/template"
    if card["max_len"] != MAX_LEN: print("NOTE: data was filtered for max_len", card["max_len"], "but MAX_LEN here is", MAX_LEN)
    train_raw = read_jsonl(find_file(f"scrapegraph_clean_train{SUFFIX}.jsonl"))
    val_raw   = read_jsonl(find_file(f"scrapegraph_clean_val{SUFFIX}.jsonl"))
    test_raw  = read_jsonl(find_file(f"scrapegraph_clean_test{SUFFIX}.jsonl"))
    random.Random(SEED).shuffle(train_raw)
    if NUM_TRAIN_EXAMPLES: train_raw = train_raw[:NUM_TRAIN_EXAMPLES]
    val_raw = val_raw[:VAL_N]
    test_raw = test_raw[:VAL_N]
    print(f"{len(train_raw)} train | {len(val_raw)} val | {len(test_raw)} held-out test (schemas never appear in train)")

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.add_bos_token = False
    BOS, EOS, PAD = tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id
    assert tokenizer("hi")["input_ids"][0] != BOS, "tokenizer is still auto-adding a BOS"
    CHAT_TEMPLATE_SHA = hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest()
    assert STD["chat_template_sha256"] == CHAT_TEMPLATE_SHA, "chat template differs from the standard"

    def render_prompt(text):
        return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)

    def encode(prompt_text, completion):
        # Deterministic boundary tokenization: every row tokenizes identically
        p = render_prompt(prompt_text)
        p_ids = tokenizer(p, add_special_tokens=False)["input_ids"]
        c_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
        full = p_ids + c_ids + [EOS]
        return {"input_ids": full, "labels": [-100] * len(p_ids) + c_ids + [EOS]}     # loss on the JSON answer only

    def prep(rs, name):
        out, too_long = [], 0
        for r in rs:
            e = encode(r["prompt"], r["completion"])
            if len(e["input_ids"]) > MAX_LEN: too_long += 1; continue          # dropped, NEVER truncated
            assert e["input_ids"].count(BOS) == 1 and e["labels"][-1] == EOS and any(l != -100 for l in e["labels"])
            out.append({**r, "enc": e})
        print(f"{name}: kept {len(out)}, dropped {too_long} longer than {MAX_LEN} tokens")
        return out
    train_rows, val_rows, test_rows = prep(train_raw, "train"), prep(val_raw, "val"), prep(test_raw, "test")
    print("deterministic tokenization active across all rows")
    assert train_rows and val_rows

    L_ = np.array([len(r["enc"]["input_ids"]) for r in train_rows]); C_ = np.array([sum(l != -100 for l in r["enc"]["labels"]) for r in train_rows])
    print(f"total tokens mean {L_.mean():.0f} | p95 {np.percentile(L_, 95):.0f} | max {L_.max()}   answer tokens mean {C_.mean():.0f} | p95 {np.percentile(C_, 95):.0f}")
    e = train_rows[0]["enc"]
    print("decoded completion:", repr(tokenizer.decode([t for t in e["labels"] if t != -100])[:200]))


181 train | 7 val | 5 held-out test (schemas never appear in train)
train: kept 181, dropped 0 longer than 1536 tokens
val: kept 7, dropped 0 longer than 1536 tokens
test: kept 5, dropped 0 longer than 1536 tokens
deterministic tokenization active across all rows
total tokens mean 1056 | p95 1506 | max 1535   answer tokens mean 88 | p95 283
decoded completion: '{"base_url": "https://www.kruidvat.nl/colgate-maximum-caries-protection-tandpasta/p/5154031", "image_url": "", "title": "", "price": null, "sale_price": null, "description": null}</s>'


In [26]:
if RUN_TRAIN:
    from transformers import AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION, quantization_config=bnb_config,
                                                 device_map={"": 0}, dtype=torch.float16)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    lora_config = LoraConfig(r=SL["r"], lora_alpha=SL["alpha"], lora_dropout=SL["dropout"], bias=SL["bias"],
                             target_modules=SL["target_modules"], use_rslora=SL["use_rslora"], use_dora=SL["use_dora"], task_type="CAUSAL_LM")
    model = get_peft_model(model, lora_config)
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()
    model.print_trainable_parameters()

    mc = AutoConfig.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
    DIMS, N_LAYERS = module_dims(mc.hidden_size, mc.num_attention_heads, mc.num_key_value_heads, mc.intermediate_size), mc.num_hidden_layers
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    expected = N_LAYERS * sum(SL["r"] * (i + o) for i, o in DIMS.values())
    assert n_train == expected, f"trainable params {n_train} != expected {expected} for the standard config"
    print("trainable params match the standard:", f"{n_train:,}")


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
trainable params match the standard: 41,943,040


In [27]:
if RUN_TRAIN:
    import importlib.metadata as md
    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    REGISTRY = {}
    def write_registry(**extra):
        REGISTRY.update({
            "registry_schema": "fragment-registry/v1", "fragment_id": FRAGMENT_ID, "version": "2.0.0",
            "fragment_type": "task - structured extraction (pair with schema-constrained decoding at serving time)",
            "task": "extract information from a text as one JSON object that follows a given JSON schema",
            "prompt_contract": PROMPT_CONTRACT,
            "base_model": BASE_MODEL, "base_revision": BASE_REVISION, "chat_template_sha256": CHAT_TEMPLATE_SHA,
            "quantization": STD["quantization"], "standard_schema": STD["schema"],
            "lora": {**SL, "target_modules": sorted(SL["target_modules"])}, "lora_scaling": STD_SCALING,
            "compatibility_key": compat_key(BASE_MODEL, BASE_REVISION, SL["r"], SL["alpha"], SL["target_modules"], SL["use_rslora"], SL["use_dora"], CHAT_TEMPLATE_SHA),
            "training": {**STD["training"], "epochs_run": EPOCHS, "num_train_examples": len(train_rows), "max_len": MAX_LEN, "smoke_test": SMOKE_TEST},
            "dataset": {"name": card["dataset"], "revision": card["dataset_revision"], "data_card": card},
            "references": ["Hu et al. 2021 (LoRA)", "Schulman et al. 2025 (LoRA Without Regret): all-layer targets, alpha 32",
                           "Biderman et al. 2024 (LoRA Learns Less and Forgets Less)"],
            "adapter_file": "adapter_model.safetensors",
            "env": {k: md.version(k) for k in ("transformers", "peft", "torch", "bitsandbytes", "datasets")},
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        })
        if os.path.exists(f"{ADAPTER_DIR}/adapter_model.safetensors"):
            REGISTRY["adapter_size_bytes"] = os.path.getsize(f"{ADAPTER_DIR}/adapter_model.safetensors")
            REGISTRY["sha256"] = sha256_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
        REGISTRY.update(extra)
        if os.path.isdir(ADAPTER_DIR):
            with open(f"{ADAPTER_DIR}/registry_entry.json", "w") as f:
                json.dump(REGISTRY, f, indent=2, default=lambda o: sorted(o) if isinstance(o, set) else str(o))
    print("helpers ok (registry helper ready)")


helpers ok (registry helper ready)


In [28]:
if RUN_TRAIN:
    from transformers import Trainer, TrainingArguments
    from transformers.trainer_utils import get_last_checkpoint

    class DS(torch.utils.data.Dataset):
        def __init__(self, rs): self.x = [r["enc"] for r in rs]
        def __len__(self): return len(self.x)
        def __getitem__(self, i): return self.x[i]

    def collate(feats):
        n = max(len(f["input_ids"]) for f in feats)
        ids = torch.full((len(feats), n), PAD, dtype=torch.long)
        lab = torch.full((len(feats), n), -100, dtype=torch.long)
        att = torch.zeros((len(feats), n), dtype=torch.long)
        for i, f in enumerate(feats):
            k = len(f["input_ids"])
            ids[i, :k] = torch.tensor(f["input_ids"]); lab[i, :k] = torch.tensor(f["labels"]); att[i, :k] = 1
        return {"input_ids": ids, "attention_mask": att, "labels": lab}

    steps_per_epoch = math.ceil(len(train_rows) / (BATCH * ACCUM))
    total_steps = MAX_STEPS if MAX_STEPS > 0 else steps_per_epoch * EPOCHS
    T = STD["training"]
    desired_args = dict(
        output_dir=CKPT_DIR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS, max_steps=MAX_STEPS, learning_rate=T["lr"], lr_scheduler_type=T["scheduler"],
        warmup_steps=max(1, int(T["warmup_frac"] * total_steps)), weight_decay=T["weight_decay"], optim=T["optimizer"],
        fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1 if SMOKE_TEST else 10, eval_strategy="steps", eval_steps=EVAL_EVERY, save_strategy="steps", save_steps=EVAL_EVERY,
        save_total_limit=T["save_total_limit"], load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        remove_unused_columns=False, report_to="none", seed=SEED, dataloader_num_workers=0)
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
    # Ensure backward compatibility between eval_strategy and evaluation_strategy
    if "eval_strategy" not in accepted and "evaluation_strategy" in accepted:
        desired_args["evaluation_strategy"] = desired_args.pop("eval_strategy")
    rejected = [k for k in desired_args if k not in accepted]
    if rejected: print("TrainingArguments rejected on this install (dropped):", rejected)
    assert "load_best_model_at_end" in accepted, "Transformers version must support load_best_model_at_end"
    args = TrainingArguments(**{k: v for k, v in desired_args.items() if k in accepted})

    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()
    trainer = Trainer(model=model, args=args, train_dataset=DS(train_rows), eval_dataset=DS(val_rows), data_collator=collate)
    print(f"steps/epoch {steps_per_epoch} | total steps {total_steps} | warmup {desired_args['warmup_steps']}")

    base_val = trainer.evaluate()
    print("BASE (untrained) validation loss:", round(base_val["eval_loss"], 4))
    last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
    print("resuming from:", last)
    out = trainer.train(resume_from_checkpoint=last)
    print([(h["step"], h.get("loss"), h.get("grad_norm"), h.get("learning_rate")) for h in trainer.state.log_history if "loss" in h])

    if SMOKE_TEST:
        sec = out.metrics["train_runtime"] / max(1, MAX_STEPS)
        print(f"\nSMOKE TIMING (upper bound, includes eval): {sec:.1f}s per optimizer step. "
              f"Real run hours ~ {sec:.0f}s x (training rows / 16 x {STD['training']['epochs']} epochs) / 3600")

    best_val = trainer.evaluate()
    print("BEST-checkpoint validation loss:", round(best_val["eval_loss"], 4), "| best ckpt:", trainer.state.best_model_checkpoint)
    test_eval = trainer.evaluate(eval_dataset=DS(test_rows))
    print("HELD-OUT TEST loss (unseen schemas):", round(test_eval["eval_loss"], 4))
    model.save_pretrained(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
    write_registry(base_val_loss=base_val["eval_loss"], best_val_loss=best_val["eval_loss"],
                   test_loss=test_eval["eval_loss"], best_checkpoint=str(trainer.state.best_model_checkpoint))
    print("adapter + registry saved:", os.listdir(ADAPTER_DIR))


steps/epoch 12 | total steps 24 | warmup 1


BASE (untrained) validation loss: 0.6875
resuming from: None


Step,Training Loss,Validation Loss


[(10, 0.18280690908432007, 0.551365077495575, 0.0001391304347826087), (20, 0.037443962693214414, 0.5361936092376709, 5.217391304347826e-05)]
BEST-checkpoint validation loss: 0.1212 | best ckpt: None
HELD-OUT TEST loss (unseen schemas): 0.0388
adapter + registry saved: ['adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'tokenizer.json', 'chat_template.jinja', 'registry_entry.json', 'adapter_config.json']


In [42]:
if RUN_TRAIN:
    rows, verdict = check_adapter_dir(ADAPTER_DIR, DIMS, N_LAYERS)
    assert verdict in ("MERGE-READY", "COMPATIBLE (some checks unverified)"), verdict
    from safetensors.torch import load_file
    w = load_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
    assert len(w) == N_LAYERS * len(SL["target_modules"]) * 2, f"expected {N_LAYERS * len(SL['target_modules']) * 2} tensors, got {len(w)}"
    assert all(torch.isfinite(t).all() for t in w.values()), "NaN/Inf in adapter"
    assert any((t != 0).any() for k, t in w.items() if "lora_B" in k), "all lora_B are zero - nothing was learned"
    norms = lora_delta_norms(ADAPTER_DIR)
    print("delta norms (scale of this fragment's task vector):", {k: round(v, 3) if isinstance(v, float) else v for k, v in norms.items()})
    write_registry(delta_norms=norms, standard_check=verdict)
    entry = json.load(open(f"{ADAPTER_DIR}/registry_entry.json"))
    assert entry["sha256"] == sha256_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
    print("registry OK:", {k: entry[k] for k in ["fragment_id", "compatibility_key", "base_revision", "adapter_size_bytes", "sha256"]})
    if not SMOKE_TEST:
        shutil.make_archive(f"{OUT_ROOT}/json-extract-lora-adapter", "zip", ADAPTER_DIR)
        print("zipped ->", f"{OUT_ROOT}/json-extract-lora-adapter.zip")
    print("\nTRAINING SMOKE PASSED." if SMOKE_TEST
          else "\nDONE. Next: notebook 3 (evaluation). The real quality check happens there, not in the loss.")



=== json-extract-lora-adapter ===
  [OK       ] peft_type is LORA                          LORA
  [OK       ] base model                                 mistralai/Mistral-7B-Instruct-v0.2
  [OK       ] rank r                                     16 (standard 16)
  [OK       ] lora_alpha / scaling                       alpha 32, scaling 2 (standard 2)
  [OK       ] target_modules                             ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
  [OK       ] use_dora                                   
  [OK       ] use_rslora                                 False
  [OK       ] no rank/alpha patterns                     
  [OK       ] no modules_to_save (extra full weights)    None
  [OK       ] only LoRA A/B tensors                      []
  [OK       ] tensor shapes                              []
  [OK       ] coverage (all layers x modules)            448/448 tensors
  [OK       ] base revision                              63a8b081895390a26e140

## Memory handover to evaluation

In [43]:
# free the training model before evaluation loads a fresh copy (two 7B models must not share the T4)
for _n in ("trainer", "model", "lora_config", "out", "base_val", "best_val", "w"):
    globals().pop(_n, None)
import gc, torch
gc.collect(); torch.cuda.empty_cache()
used = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory still allocated: {used:.1f} GB")
if used > 2.5: print("WARNING: memory was not released. Restart the kernel and run with RUN_DATA = False, RUN_TRAIN = False (stage 3 loads the adapter from disk).")


GPU memory still allocated: 0.5 GB


## Stage 3: evaluation against fair baselines, then freeze the reference scores

In [44]:
if RUN_EVAL:
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    import json, re, math, time, hashlib, random, glob, contextlib
    from collections import Counter, defaultdict
    import numpy as np
    import torch
    assert torch.cuda.is_available() and torch.cuda.device_count() == 1

    FRAGMENT_ID     = "json-extract-lora"
    ADAPTER_DIR     = None     # None = auto-find the adapter folder whose registry fragment_id is FRAGMENT_ID. Or paste the path.
    OLD_ADAPTER_DIR = None     # optional: folder of the old attention-only adapter (q/k/v), for the target-modules ablation
    N_PER_TOPIC     = 1 if SMOKE_TEST else 10   # 10 rows per paraloq topic (8 topics -> ~80 eval rows total for tight CIs)
    GEN_MAX_NEW     = 400
    MAX_LEN         = 1536
    BASE_MODEL, BASE_REVISION = STD["base_model"], STD["base_revision"]
    OUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    _p = f"{OUT_ROOT}/json-extract-lora-adapter{SUFFIX}"
    if ADAPTER_DIR is None and os.path.isdir(_p): ADAPTER_DIR = _p     # the adapter stage 2 just wrote

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    def find_adapter(frag_id):
        paths = glob.glob("/kaggle/input/**/registry_entry.json", recursive=True) + glob.glob(f"{OUT_ROOT}/**/registry_entry.json", recursive=True)
        for p in paths:
            d = os.path.dirname(p)
            if d.endswith("-smoke"): continue
            try:
                if json.load(open(p)).get("fragment_id") == frag_id: return d
            except Exception: pass
        return None

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_bos_token = False
    EOS = tokenizer.eos_token_id
    assert hashlib.sha256((tokenizer.chat_template or "").encode()).hexdigest() == STD["chat_template_sha256"], "chat template differs from the standard"
    def render_prompt(text):
        return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
    print("config ok")


config ok


In [45]:
if RUN_EVAL:
    # ---- the frozen evaluation set. Raw fields are stored; prompts are rebuilt from the ONE template each run ----
    from datasets import load_dataset
    from jsonschema.validators import validator_for
    EVAL_FILE = f"{OUT_ROOT}/json_eval_set_v2.json"
    prev = glob.glob("/kaggle/input/**/json_eval_set_v2.json", recursive=True) + glob.glob(EVAL_FILE)
    if prev:
        raw_eval = json.load(open(prev[0])); print("loaded saved eval set:", prev[0], len(raw_eval), "rows")
    else:
        pq = load_dataset("paraloq/json_data_extraction", split="train")
        cand = []
        for r in pq:
            try: schema, item = json.loads(r["schema"]), json.loads(r["item"])
            except Exception: continue
            if not isinstance(item, dict) or not (200 <= len(r["text"]) <= 3000): continue
            try: validator_for(schema).check_schema(schema)
            except Exception: continue
            cand.append({"schema": schema, "text": r["text"], "item": item, "topic": r["topic"], "title": r["title"]})
        by = defaultdict(list)
        for x in cand: by[x["topic"]].append(x)
        rng, raw_eval = random.Random(0), []
        for t, xs in by.items(): rng.shuffle(xs); raw_eval += xs[:12]
        json.dump(raw_eval, open(EVAL_FILE, "w")); print("built eval set:", len(raw_eval), "rows (saved json_eval_set_v2.json)")

    def build_row(x):
        s = json.dumps(x["schema"], separators=(",", ":"))
        return {"prompt": TEMPLATE.format(instruction=DEFAULT_INSTRUCTION, schema=s, content=x["text"]), "ref": x["item"],
                "topic": x["topic"], "schema_obj": x["schema"], "schema_str": s}
    pool = [build_row(x) for x in raw_eval]
    pool = [r for r in pool if len(tokenizer(render_prompt(r["prompt"]), add_special_tokens=False)["input_ids"]) <= MAX_LEN - 400]
    per_topic = defaultdict(list)
    for r in pool:
        if len(per_topic[r["topic"]]) < N_PER_TOPIC: per_topic[r["topic"]].append(r)
    eval_set = [per_topic[t][k] for k in range(N_PER_TOPIC) for t in per_topic if k < len(per_topic[t])]   # round-robin: a partial run still covers every topic
    EVAL_SHA = hashlib.sha256(json.dumps([r["prompt"] for r in eval_set]).encode()).hexdigest()
    print(f"{len(eval_set)} rows across {len(per_topic)} topics:", {t: len(v) for t, v in per_topic.items()})


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data.parquet: reconstructing file:   0%|          |  0.00B /  967kB            

data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/484 [00:00<?, ? examples/s]

/tmp/ipykernel_49/94372866.py:16: DeprecationWarning: The metaschema specified by $schema was not found. Using the latest draft to validate, but this will raise an error in the future.
  try: validator_for(schema).check_schema(schema)


built eval set: 96 rows (saved json_eval_set_v2.json)
71 rows across 8 topics: {'medical': 7, 'ecommerce': 10, 'manufacturing': 10, 'business': 6, 'simple': 10, 'media': 10, 'travel': 10, 'technology': 8}


In [46]:
if RUN_EVAL:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
    from peft import PeftModel

    ADAPTER_DIR = ADAPTER_DIR or find_adapter(FRAGMENT_ID)
    assert ADAPTER_DIR, "adapter not found: attach notebook 2's output (Add Input) or paste the folder path into ADAPTER_DIR"
    print("adapter folder:", ADAPTER_DIR)
    rows, verdict = check_adapter_dir(ADAPTER_DIR)
    assert verdict in ("MERGE-READY", "COMPATIBLE (some checks unverified)"), verdict
    reg = json.load(open(f"{ADAPTER_DIR}/registry_entry.json"))
    assert reg["sha256"] == sha256_file(f"{ADAPTER_DIR}/adapter_model.safetensors"), "adapter file does not match its registry hash"

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION, quantization_config=bnb, device_map={"": 0}, dtype=torch.float16)
    model = PeftModel.from_pretrained(base, ADAPTER_DIR, adapter_name="default")
    CONDS = ["base", "lora"]
    if OLD_ADAPTER_DIR:
        model.load_adapter(OLD_ADAPTER_DIR, adapter_name="qkv"); CONDS.append("lora_qkv")
    model.eval(); model.config.use_cache = True
    print("loaded. conditions:", CONDS)


adapter folder: /kaggle/working/json-extract-lora-adapter

=== json-extract-lora-adapter ===
  [OK       ] peft_type is LORA                          LORA
  [OK       ] base model                                 mistralai/Mistral-7B-Instruct-v0.2
  [OK       ] rank r                                     16 (standard 16)
  [OK       ] lora_alpha / scaling                       alpha 32, scaling 2 (standard 2)
  [OK       ] target_modules                             ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
  [OK       ] use_dora                                   
  [OK       ] use_rslora                                 False
  [OK       ] no rank/alpha patterns                     
  [OK       ] no modules_to_save (extra full weights)    None
  [OK       ] only LoRA A/B tensors                      []
  [OK       ] tensor shapes                              []
  [OK       ] coverage (all layers x modules)            448/448 tensors
  [OK       ] base r

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded. conditions: ['base', 'lora']


In [47]:
if RUN_EVAL:
    # ---------------- parsing ----------------
    def strip_fence(s): return re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", s.strip()).strip()
    def parse_strict(s):
        # the WHOLE output must be one JSON object (after removing a code fence). Measures learned format discipline.
        try: o = json.loads(strip_fence(s))
        except Exception: return None
        return o if isinstance(o, dict) else None
    def first_object(s):
        # lenient: the first balanced, parseable {...} anywhere in the text (string-aware). Used for scoring every condition.
        start = s.find("{")
        while start != -1:
            depth, in_str, esc = 0, False, False
            for i in range(start, len(s)):
                ch = s[i]
                if in_str:
                    if esc: esc = False
                    elif ch == "\\": esc = True
                    elif ch == '"': in_str = False
                elif ch == '"': in_str = True
                elif ch == "{": depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        try:
                            o = json.loads(s[start:i + 1])
                            if isinstance(o, dict): return o
                        except Exception: pass
                        break
            start = s.find("{", start + 1)
        return None

    # ---------------- metric: multiset field-level P/R/F1 over (path, normalised leaf value) ----------------
    def nv(v): return re.sub(r"\s+", " ", v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)).strip().lower()
    def flat(o, path=""):
        out = Counter()
        if isinstance(o, dict):
            for k, v in o.items(): out += flat(v, f"{path}.{k}")
        elif isinstance(o, list):
            for v in o: out += flat(v, f"{path}[]")
        elif o is not None and o != "": out[(path, nv(o))] += 1
        return out
    def prf(pred, ref):
        p, r = flat(pred), flat(ref); tp = sum((p & r).values())
        P = tp / sum(p.values()) if p else 0.0; R = tp / sum(r.values()) if r else 0.0
        return P, R, (2 * P * R / (P + R) if P + R else 0.0)
    def schema_ok(obj, schema):
        if obj is None or schema is None: return False
        try:
            from jsonschema import validate
            validate(obj, schema); return True
        except Exception: return False
    def score(text, ref, n_gen, schema=None):
        strict, lenient = parse_strict(text), first_object(text)
        P, R, F = prf(lenient, ref) if lenient is not None else (0.0, 0.0, 0.0)
        return {"strict": strict is not None, "lenient": lenient is not None, "P": P, "R": R, "F1": F,
                "trunc": bool(lenient is None and n_gen >= GEN_MAX_NEW), "schema_ok": schema_ok(lenient, schema)}

    # ---------------- generation: stop as soon as the first top-level JSON object closes (string-aware, incremental) ----------------
    class JsonDone(StoppingCriteria):
        def __init__(self, tok, start):
            self.tok, self.start = tok, start
            self.pos = 0; self.depth = 0; self.seen = False; self.in_str = False; self.esc = False; self.done = False
        def __call__(self, input_ids, scores, **kw):
            if not self.done:
                text = self.tok.decode(input_ids[0][self.start:], skip_special_tokens=True)
                for ch in text[self.pos:]:
                    if self.in_str:
                        if self.esc: self.esc = False
                        elif ch == "\\": self.esc = True
                        elif ch == '"': self.in_str = False
                    elif ch == '"': self.in_str = True
                    elif ch == "{": self.depth += 1; self.seen = True
                    elif ch == "}":
                        self.depth -= 1
                        if self.seen and self.depth == 0: self.done = True; break
                self.pos = len(text)
            return torch.full((input_ids.shape[0],), self.done, dtype=torch.bool, device=input_ids.device)

    def gen_plain(r, cond):
        inp = tokenizer(render_prompt(r["prompt"]), return_tensors="pt", add_special_tokens=False).to(model.device)
        n = inp["input_ids"].shape[1]
        if cond == "lora": model.set_adapter("default")
        if cond == "lora_qkv": model.set_adapter("qkv")
        ctx = model.disable_adapter() if cond == "base" else contextlib.nullcontext()
        with torch.no_grad(), ctx:
            out = model.generate(**inp, max_new_tokens=GEN_MAX_NEW, do_sample=False, pad_token_id=EOS,
                                 stopping_criteria=StoppingCriteriaList([JsonDone(tokenizer, n)]))
        ids = out[0][n:]
        return tokenizer.decode(ids, skip_special_tokens=True), int(ids.shape[0])

    # ---------------- resumable runner ----------------
    RES_PATH = f"{OUT_ROOT}/json_eval_results{SUFFIX}.jsonl"
    res = {}
    if os.path.exists(RES_PATH):
        for l in open(RES_PATH): d = json.loads(l); res[(d["cond"], d["i"])] = d
    print("saved results found:", len(res))

    def run_all(conds, fn, tolerate=False):
        t0, fresh = time.time(), 0
        with open(RES_PATH, "a") as f:
            for i, r in enumerate(eval_set):
                for c in conds:
                    if (c, i) in res: continue
                    try:
                        text, ng = fn(r, c)
                        d = {"cond": c, "i": i, "topic": r["topic"], **score(text, r["ref"], ng, r["schema_obj"]), "out": text[:1500]}
                    except Exception as e:
                        if not tolerate: raise
                        d = {"cond": c, "i": i, "topic": r["topic"], "error": repr(e)[:300], "strict": False, "lenient": False,
                             "P": 0.0, "R": 0.0, "F1": 0.0, "trunc": False, "schema_ok": False, "out": ""}
                    res[(c, i)] = d; f.write(json.dumps(d) + "\n"); f.flush(); fresh += 1
                if (i + 1) % 5 == 0:
                    msg = " | ".join(f"{c} F1 {np.mean([res[(c, j)]['F1'] for j in range(i + 1) if (c, j) in res and 'error' not in res[(c, j)]] or [0]):.3f}" for c in conds)
                    print(f"{i+1}/{len(eval_set)} | {(time.time() - t0) / max(1, fresh):.0f}s/generation | {msg}", flush=True)
    print("utilities ready")


saved results found: 0
utilities ready


In [48]:
if RUN_EVAL:
    run_all(CONDS, gen_plain)       # base vs lora (+ lora_qkv if OLD_ADAPTER_DIR was set). Resumable: re-run this cell after a disconnect.
    print("done:", {c: sum(1 for k in res if k[0] == c) for c in CONDS})


5/71 | 23s/generation | base F1 0.367 | lora F1 0.449
10/71 | 24s/generation | base F1 0.321 | lora F1 0.500
15/71 | 25s/generation | base F1 0.363 | lora F1 0.628
20/71 | 25s/generation | base F1 0.415 | lora F1 0.706
25/71 | 24s/generation | base F1 0.478 | lora F1 0.734
30/71 | 24s/generation | base F1 0.459 | lora F1 0.715
35/71 | 24s/generation | base F1 0.472 | lora F1 0.706
40/71 | 24s/generation | base F1 0.482 | lora F1 0.715
45/71 | 23s/generation | base F1 0.507 | lora F1 0.716
50/71 | 23s/generation | base F1 0.528 | lora F1 0.731
55/71 | 23s/generation | base F1 0.519 | lora F1 0.715
60/71 | 23s/generation | base F1 0.509 | lora F1 0.729
65/71 | 22s/generation | base F1 0.532 | lora F1 0.734
70/71 | 22s/generation | base F1 0.506 | lora F1 0.692
done: {'base': 71, 'lora': 71}


In [49]:
if RUN_EVAL:
    def boot_ci(x, B=2000, seed=0):
        x = np.asarray(x, float); rng = np.random.default_rng(seed)
        m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]
        return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

    def rows_of(c): return {i: res[(c, i)] for i in range(len(eval_set)) if (c, i) in res and "error" not in res[(c, i)]}

    def summarize(conds):
        out = {}
        print(f"{'condition':<10}{'n':>4} {'raw=only JSON':>14} {'JSON found':>11} {'schema-valid':>13} {'P':>6} {'R':>6} {'F1':>6}  {'95% CI':<15}{'cut-off':>8}{'errors':>7}")
        for c in conds:
            rs = rows_of(c)
            if not rs: continue
            v = list(rs.values()); f1 = [x["F1"] for x in v]; lo, hi = boot_ci(f1)
            errs = sum(1 for (cc, i), d in res.items() if cc == c and "error" in d)
            out[c] = {"n": len(v), "raw_only_json": float(np.mean([x["strict"] for x in v])), "json_found": float(np.mean([x["lenient"] for x in v])), "schema_valid": float(np.mean([bool(x.get("schema_ok")) for x in v])),
                      "P": float(np.mean([x["P"] for x in v])), "R": float(np.mean([x["R"] for x in v])), "F1": float(np.mean(f1)),
                      "F1_ci95": [lo, hi], "cut_off": float(np.mean([x["trunc"] for x in v])), "errors": errs}
            o = out[c]
            print(f"{c:<10}{o['n']:>4} {o['raw_only_json']:>13.0%} {o['json_found']:>11.0%} {o['schema_valid']:>13.0%} {o['P']:>6.3f} {o['R']:>6.3f} {o['F1']:>6.3f}  [{lo:.3f},{hi:.3f}]  {o['cut_off']:>7.0%}{errs:>7}")
        return out

    def paired(a, b):
        ra, rb = rows_of(a), rows_of(b); ids = sorted(set(ra) & set(rb))
        if len(ids) < 5: return None
        d = np.array([ra[i]["F1"] - rb[i]["F1"] for i in ids]); lo, hi = boot_ci(d)
        return {"n": len(ids), "mean_diff": float(d.mean()), "ci95": [lo, hi]}

    ALL = [c for c in ["base", "lora", "lora_qkv", "base_gcd", "lora_gcd"] if any(k[0] == c for k in res)]
    summary = summarize(ALL)
    print()
    for a, b in [("lora", "base"), ("lora", "lora_qkv"), ("lora_gcd", "base_gcd"), ("lora_gcd", "lora"), ("lora_gcd", "lora_qkv")]:
        if a in summary and b in summary:
            p = paired(a, b)
            if p: print(f"paired F1 difference {a} - {b}: {p['mean_diff']:+.3f}  95% CI [{p['ci95'][0]:+.3f}, {p['ci95'][1]:+.3f}]  (n={p['n']})  ->", "CLEAR" if p["ci95"][0] > 0 or p["ci95"][1] < 0 else "NOT distinguishable from zero")
    print("\nper-topic F1:")
    for c in ALL:
        rs = rows_of(c)
        print(f"  {c:<9}", {t: round(float(np.mean([x['F1'] for x in rs.values() if x['topic'] == t] or [0])), 2) for t in per_topic})
    bad = [d for d in rows_of("lora").values() if not d["lenient"]][:3]
    for d in bad: print("\nADAPTER FAILURE (no parseable JSON), topic", d["topic"], "| tail of output:\n", d["out"][-300:])


condition    n  raw=only JSON  JSON found  schema-valid      P      R     F1  95% CI          cut-off errors
base        71           75%         99%           65%  0.523  0.506  0.510  [0.407,0.602]       1%      0
lora        71           89%         97%           76%  0.715  0.690  0.695  [0.608,0.774]       3%      0

paired F1 difference lora - base: +0.185  95% CI [+0.097, +0.276]  (n=71)  -> CLEAR

per-topic F1:
  base      {'medical': 0.31, 'ecommerce': 0.44, 'manufacturing': 0.76, 'business': 0.34, 'simple': 0.27, 'media': 0.48, 'travel': 0.56, 'technology': 0.85}
  lora      {'medical': 0.66, 'ecommerce': 0.59, 'manufacturing': 0.7, 'business': 0.59, 'simple': 0.57, 'media': 0.7, 'travel': 0.9, 'technology': 0.83}

ADAPTER FAILURE (no parseable JSON), topic manufacturing | tail of output:
 y and found to be functioning as expected.", "changeVerificationResults": "The new conveyor belt system has been in operation for two weeks and has met all expectations. The system is funct

In [50]:
if RUN_EVAL:
    # OPTIONAL: grammar-constrained decoding (base_gcd, lora_gcd) on a SUBSET of the eval rows. Resumable: re-run to continue.
    N_GCD = 24          # 24 rows = 3 per topic (eval_set is in round-robin topic order). Raise it only after you see the seconds-per-row below.
    try:
        try:
            import outlines
        except ImportError:
            import subprocess, sys
            r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "outlines"], capture_output=True, text=True)
            print("pip outlines:", "OK" if r.returncode == 0 else r.stderr[-800:])
            import outlines
        from outlines.types import JsonSchema
        try: model.base_model.enable_adapter_layers()          # in case an earlier interrupt left the adapter switched off
        except Exception as e: print("could not re-enable adapters:", repr(e))
        try: model.generation_config.pad_token_id = EOS        # silences the repeated "Setting pad_token_id" message
        except Exception: pass
        o_model = outlines.from_transformers(model.get_base_model(), tokenizer)   # the LoRA layers live inside this module; adapter on/off is toggled on `model`
        def clean_schema_for_outlines(raw_s):
            """Outlines crashes on $schema, $id, title, etc. Strip them recursively."""
            try: d = json.loads(raw_s) if isinstance(raw_s, str) else dict(raw_s)
            except Exception: return raw_s
            def _strip(o):
                if isinstance(o, dict):
                    for k in ("$schema", "$id", "id", "title", "description", "$comment"): o.pop(k, None)
                    for v in o.values(): _strip(v)
                elif isinstance(o, list):
                    for v in o: _strip(v)
                return o
            return json.dumps(_strip(d), separators=(",", ":"))

        _gens = {}
        def gen_gcd(r, cond):
            s_outlines = clean_schema_for_outlines(r["schema_str"])
            if s_outlines not in _gens:
                _gens[s_outlines] = outlines.Generator(o_model, JsonSchema(s_outlines))
            if cond == "lora_gcd": model.set_adapter("default")
            ctx = model.disable_adapter() if cond == "base_gcd" else contextlib.nullcontext()
            with ctx:
                text = _gens[s_outlines](render_prompt(r["prompt"]), max_new_tokens=GEN_MAX_NEW)
            return text, len(tokenizer(text, add_special_tokens=False)["input_ids"])
        with open(RES_PATH, "a") as f:
            for i, r in enumerate(eval_set[:N_GCD]):
                for c in ("base_gcd", "lora_gcd"):
                    if (c, i) in res: continue
                    t1 = time.time()
                    try:
                        text, ng = gen_gcd(r, c)
                        d = {"cond": c, "i": i, "topic": r["topic"], **score(text, r["ref"], ng, r["schema_obj"]), "out": text[:1500]}
                    except Exception as e:
                        d = {"cond": c, "i": i, "topic": r["topic"], "error": repr(e)[:300], "strict": False, "lenient": False,
                             "P": 0.0, "R": 0.0, "F1": 0.0, "trunc": False, "schema_ok": False, "out": ""}
                    res[(c, i)] = d; f.write(json.dumps(d) + "\n"); f.flush()
                    print(f"row {i+1}/{N_GCD} {c:<9} {time.time() - t1:5.0f}s | " +
                          (f"ERROR {d['error']}" if "error" in d else f"F1 {d['F1']:.2f} | schema_ok {d['schema_ok']}"), flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); print("GCD cell failed:", repr(e))
    for _c in ("lora", "lora_gcd"):
        if (_c, 0) in res: print(_c, "row 0 output:", res[(_c, 0)]["out"][:260].replace("\n", " "))


pip outlines: OK
row 1/24 base_gcd     19s | F1 0.45 | schema_ok False
row 1/24 lora_gcd     22s | F1 0.45 | schema_ok False
row 2/24 base_gcd      5s | F1 0.31 | schema_ok True
row 2/24 lora_gcd      8s | F1 0.31 | schema_ok True
row 3/24 base_gcd     15s | F1 0.97 | schema_ok True
row 3/24 lora_gcd     24s | F1 0.97 | schema_ok True
row 4/24 base_gcd     27s | F1 0.00 | schema_ok False
row 4/24 lora_gcd     47s | F1 0.00 | schema_ok False
row 5/24 base_gcd     16s | F1 0.45 | schema_ok False
row 5/24 lora_gcd     25s | F1 0.45 | schema_ok False
row 6/24 base_gcd     17s | F1 0.86 | schema_ok False
row 6/24 lora_gcd     23s | F1 0.86 | schema_ok False
row 7/24 base_gcd      9s | F1 1.00 | schema_ok True
row 7/24 lora_gcd     14s | F1 1.00 | schema_ok True
row 8/24 base_gcd      0s | ERROR ValueError('Unsupported JSON Schema structure {"$defs":{"CloudServiceSubscription":{"type":"object","required":["service","subscriptionId"],"properties":{"service":{"type":"string","enum":["GKE Hub",

In [51]:
if RUN_EVAL:
    summary = summarize([c for c in ["base", "lora", "lora_qkv", "base_gcd", "lora_gcd"] if any(k[0] == c for k in res)])
    frozen = {"fragment_id": FRAGMENT_ID, "adapter_sha256": reg["sha256"], "eval_set_sha256": EVAL_SHA, "n_rows": len(eval_set),
              "topics": {t: len(v) for t, v in per_topic.items()}, "template": TEMPLATE, "default_instruction": DEFAULT_INSTRUCTION,
              "prompt_contract": PROMPT_CONTRACT,
              "generation": {"do_sample": False, "max_new_tokens": GEN_MAX_NEW, "stop": "first complete top-level JSON object"},
              "parse": {"scoring": "lenient: first balanced parseable JSON object", "raw_only_json": "whole output must be one JSON object"},
              "metric": "multiset field-level precision/recall/F1 over (path, normalised leaf value); empty/null leaves ignored; no JSON found = 0",
              "results": summary}
    json.dump(frozen, open(f"{OUT_ROOT}/json_reference{SUFFIX}.json", "w"), indent=2)
    print("wrote json_reference.json. Every merged model must be scored with THIS eval set, THIS code and THESE settings.")
    reg["eval_scores"] = {"json_reference": summary, "eval_set_sha256": EVAL_SHA}
    try:
        json.dump(reg, open(f"{ADAPTER_DIR}/registry_entry.json", "w"), indent=2); print("registry updated in place")
    except Exception:
        json.dump(reg, open(f"{OUT_ROOT}/registry_entry_with_scores.json", "w"), indent=2); print("adapter folder is read-only: wrote registry_entry_with_scores.json instead")


condition    n  raw=only JSON  JSON found  schema-valid      P      R     F1  95% CI          cut-off errors
base        71           75%         99%           65%  0.523  0.506  0.510  [0.407,0.602]       1%      0
lora        71           89%         97%           76%  0.715  0.690  0.695  [0.608,0.774]       3%      0
base_gcd    19           89%        100%           58%  0.734  0.668  0.697  [0.553,0.820]       0%      5
lora_gcd    19           95%        100%           58%  0.803  0.729  0.762  [0.630,0.870]       0%      5
wrote json_reference.json. Every merged model must be scored with THIS eval set, THIS code and THESE settings.
registry updated in place
